In [ ]:
import os

In [ ]:
class BasePipeline:
    '''
    Classe base para pipelines de dados, fornecendo métodos comuns como acesso a segredos.
     - Busca segredos em variáveis de ambiente no arquivo config_env.py (Databricks Community Edition não permite uso de dbutils.secrets).
     - Permite que pipelines herdem e utilizem esses métodos.
    '''
    def __init__(self, dominio):
        # Não usamos mais dbutils.secrets
        self.db_user = os.getenv(f"{dominio}_SUPABASE_USER")
        self.db_pass = os.getenv(f"{dominio}_SUPABASE_PASS")
        self.db_host = os.getenv(f"{dominio}_SUPABASE_HOST")
        self.db_name = os.getenv(f"{dominio}_SUPABASE_DB")

    def get_jdbc_url(self):
        return f"jdbc:postgresql://{self.db_host}:5432/{self.db_name}"
    
    def get_connection_properties(self):
        return {
            "user": self.db_user,
            "password": self.db_pass,
            "driver": "org.postgresql.Driver",
            "ssl": "true",
            "sslmode": "require"
        }

In [ ]:
class BronzePipeline(BasePipeline):
    _dominios_validos = ('rh', 'financas' , 'marketing')
    def __init__(self, dominio: str):
        if dominio not in self._dominios_validos:
            raise ValueError(
                f"Domínio '{dominio}' inválido. "
                f"Use um de: {sorted(self._dominios_validos)}"
            )
        super().__init__(dominio=dominio.upper())
        self.dominio = dominio
        self.schema = 'bronze'
        self.catalog = f'{self.dominio}_prod'

    def extract_from_postgres(self, table_name):
        """Lê os dados brutos do Supabase via JDBC"""
        print(f"Extraindo dados de: {table_name}...")
        
        return spark.read.jdbc(
            url=self.get_jdbc_url(),
            table=table_name,
            properties=self.get_connection_properties()
        )
    
    def load_to_bronze(self, df, table_name):
        """Salva os dados no formato Delta na camada Bronze"""
        full_table_path = f"{self.catalog}.{self.schema}.{table_name}"
        
        print(f"Salvando na Bronze: {full_table_path}...")
        
        df.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(full_table_path)
        
        print("Carga Bronze finalizada com sucesso!")

    def run(self, source_table):
        """Executa o pipeline completo da Bronze"""
        df_raw = self.extract_from_postgres(source_table)
        self.load_to_bronze(df_raw, source_table)

In [ ]:
class SilverPipeline:
    # TODO: Implementar a pipeline Silver
    pass

In [ ]:
class GoldPipeline:
    # TODO: Implementar a pipeline Gold
    pass